In [2]:
import os
import glob
os.environ['MPLCONFIGDIR'] = "/mnt/storage6/grace/plt_temp/"

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import xarray as xr
import netCDF4 as nc

from xarray.groupers import UniqueGrouper
from xarray.groupers import SeasonGrouper

from matplotlib.ticker import MaxNLocator

import datetime
import cftime
import time

import gsw

mask_file = xr.open_dataset('/mnt/storage4/grace/grace/data/model_masks/ANHA4_mesh_mask.nc')

In [3]:
def time_bounds(startyear, startmonth, startday, endyear, endmonth, endday):
    return [startyear, startmonth, startday, endyear, endmonth, endday]

def get_times(dates):
    start_time = datetime.date(dates[0], dates[1], dates[2])
    end_time = datetime.date(dates[3], dates[4], dates[5])

    #figure out all the dates we have model files
    delta = end_time - start_time
    times = []

    i = 0
    while i < delta.days+1:
        t = start_time + datetime.timedelta(days=i)
        if t.month == 2 and t.day == 29:
            t = datetime.date(t.year, 3, 1)
            i = i+6
        else:
            i = i+5
        times.append(t)

    return times

def preprocess_t(ds):

    ds = ds[['sossheig', 'somxlts','e3t']]
    ds = ds.sel(y_grid_T = slice(250,450))
    ds = ds.sel(x_grid_T = slice(150,375))

    return ds

In [4]:
def get_files(path, runid, times):
    #need both the u and v components of velocity

    mdl_files_t = []
    for t in times:
        mdl_files_t.append(path+"ANHA4-"+runid+"_y"+str(t.year)+"m"+str(t.month).zfill(2)+"d"+str(t.day).zfill(2)+"_gridT.nc")
    dt = xr.open_mfdataset(mdl_files_t, data_vars='minimal', coords='minimal', compat='override', preprocess=preprocess_t)
    t = dt.resample(time_counter='ME').mean()
    return dt, t

GET TEOS DATA

In [5]:
file_pattern = '/mnt/storage5/grace/data/EGK503/*_gridT.nc'
filenames = glob.glob(file_pattern)
teos_5d = xr.open_mfdataset(filenames, data_vars='minimal', coords='minimal', compat='override', preprocess=preprocess_t)
teos_monthly = teos_5d.resample(time_counter='ME').mean()

In [ ]:
teos_monthly.to_netcdf('/mnt/storage5/grace/data/EGK503_monthly.nc')

KeyboardInterrupt: 

GET EOS DATA

In [8]:
file_pattern = '/mnt/storage5/grace/data/EGK501t/*_gridT.nc'
filenames = glob.glob(file_pattern)
eos_5d = xr.open_mfdataset(filenames, data_vars='minimal', coords='minimal', compat='override', preprocess=preprocess_t)
eos_monthly = eos_5d.resample(time_counter='ME').mean()

OSError: [Errno -101] NetCDF: HDF error: '/mnt/storage5/grace/data/EGK501t/ANHA4-EGK501t_y2008m04d15_gridT.nc'

In [ ]:
eos5d = eos_5d['somxlts'].groupby(['time_counter.year'])
eosmy = eos_monthly['somxlts'].groupby(['time_counter.year'])

In [ ]:
teos5d = teos_5d['somxlts'].groupby(['time_counter.year'])
teosmy = teos_monthly['somxlts'].groupby(['time_counter.year'])

In [ ]:
years = np.arange(2002, 2022, 1)

teos5max = np.zeros(len(years))
teosmonthmax = np.zeros(len(years))

eos5max = np.zeros(len(years))
eosmonthmax = np.zeros(len(years))

counter = 0
for year in years:
    teos5max[counter] = np.nanmax(teos5d[year])
    teosmonthmax[counter] = np.nanmax(teosmy[year])
    eos5max[counter] = np.nanmax(eos5d[year])
    eosmonthmax[counter] = np.nanmax(eosmy[year])
    counter += 1

In [ ]:
plt.plot(years, teosmonthmax)
plt.plot(years, eosmonthmax)

In [ ]:
plt.contourf(teosmy[2003][2, :,:] - eosmy[2003][2, :,:])
plt.colorbar()

In [ ]:
years = np.arange(2002,2022,1)
eos5 = np.zeros(len(years))
eos5_index = {}
teos5 = np.zeros(len(years))
teos5_index = {}

for x in years:

    emonths = eos5d[x].groupby('time_counter.month')
    tmonths = teos5d[x].groupby('time_counter.month')

    emax = np.nanmax(eos5d[x])
    tmax = np.nanmax(teos5d[x])

    eos5[x-years[0]] = emax
    teos5[x-years[0]] = tmax

    for group_name, group_ds in emonths:
        m = np.where(group_ds == emax)
        if np.size(m) != 0:
            eos5_index[str(x)] = group_name 

    for group_name, group_ds in tmonths:
        m = np.where(group_ds == tmax)
        if np.size(m) != 0:
            teos5_index[str(x)] = group_name 

In [ ]:
years = np.arange(2002,2022,1)
eosm = np.zeros(len(years))
eosm_index = {}
teosm = np.zeros(len(years))
teosm_index = {}

for x in years:

    emonths = eosmy[x].groupby('time_counter.month')
    tmonths = teosmy[x].groupby('time_counter.month')

    emax = np.nanmax(eosmy[x])
    tmax = np.nanmax(teosmy[x])

    eosm[x-years[0]] = emax
    teosm[x-years[0]] = tmax

    for group_name, group_ds in emonths:
        m = np.where(group_ds == emax)
        if np.size(m) != 0:
            eosm_index[str(x)] = group_name 

    for group_name, group_ds in tmonths:
        m = np.where(group_ds == tmax)
        if np.size(m) != 0:
            teosm_index[str(x)] = group_name 

In [ ]:
def diff_mld_seas(year, teos, eos):
    
    teos_jfm = teos[year].groupby(time_counter=SeasonGrouper(["FM", "AMJ", "JAS", "OND"]))['FM'].mean('time_counter')
    eos_jfm = eos[year].groupby(time_counter=SeasonGrouper(["FM", "AMJ", "JAS", "OND"]))['FM'].mean('time_counter')

    diff = teos_jfm - eos_jfm

    land_mask = np.ma.make_mask(mask_file['tmask'][0,0,250:450,150:375] != 0)

    diff_masked = np.where(land_mask, diff, np.nan)
    
    fig, axes = plt.subplots()
    axes.set_facecolor('grey')

    min = np.nanmin(diff_masked)
    max = np.nanmax(diff_masked)

    mm = np.nanmax([np.abs(min), np.abs(max)])
    
    b = axes.contourf(diff_masked, np.linspace(-mm, mm, 30), cmap = 'seismic')
    
    fig.colorbar(b)
    fig.show()

In [ ]:
emonths = eosmy[2018]
tmonths = teosmy[2018]

teos_ind = teosm_index[str(2018)]
eos_ind = eosm_index[str(2018)]

teos_max = tmonths[teos_ind -1, :, :]
eos_max = emonths[eos_ind -1, :, :]

print(eos_max)

diff = teos_max - eos_max

print(np.nanmax(diff.values))

land_mask = np.ma.make_mask(mask_file['tmask'][0,0,250:450,150:375] != 0)

diff_masked = np.where(land_mask, diff, np.nan)

fig, axes = plt.subplots()
axes.set_facecolor('grey')

min = np.nanmin(diff_masked)
max = np.nanmax(diff_masked)

mm = np.nanmax([np.abs(min), np.abs(max)])

b = axes.contourf(diff_masked[:,:], cmap = 'seismic')

fig.colorbar(b)
fig.show()


In [ ]:
def diff_mld_max(year, teos, t_ind, eos, e_ind):

    emonths = eos[year].groupby('time_counter.month')
    tmonths = teos[year].groupby('time_counter.month')
    
    teos_ind = t_ind[str(year)]
    eos_ind = e_ind[str(year)]

    teos_max = tmonths[teos_ind]
    eos_max = emonths[eos_ind]

    diff = teos_max - eos_max
    
    land_mask = np.ma.make_mask(mask_file['tmask'][0,0,250:450,150:375] != 0)

    diff_masked = np.where(land_mask, diff, np.nan)
    
    fig, axes = plt.subplots()
    axes.set_facecolor('grey')

    min = np.nanmin(diff)
    max = np.nanmax(diff)

    mm = np.nanmax([np.abs(min), np.abs(max)])
    
    b = axes.contourf(diff_masked, np.linspace(-mm, mm, 30), cmap = 'seismic')
    
    fig.colorbar(b)
    fig.show()

In [ ]:
diff_mld_seas(2018, f_teos, f_eos)

In [ ]:
plt.scatter(years, eosm)
plt.plot(years, eosm, label = 'EOS-80 monthly')
plt.scatter(years, teosm)
plt.plot(years, teosm, label ='TEOS-10 monthly')
plt.axvline(2013, c = 'red')
plt.axvline(2018, c = 'red')
#plt.axvline(2004, c = 'red')
#plt.axvline(2005, c = 'red')
#plt.axvline(2006, c = 'red')
#plt.axvline(2007, c = 'red')

plt.xlabel('Year')
plt.ylabel('Mixed Layer Depth (m)')
plt.title('Max Yearly Mixed Layer Depth, EOS-80 & TEOS-10 monthly avg')
plt.gca().xaxis.set_major_locator(MaxNLocator(nbins=7))
plt.legend()

In [ ]:
plt.scatter(years, eos5)
plt.plot(years, eos5, label = 'EOS-80 5 day')
plt.scatter(years, teos5)
plt.plot(years, teos5, label ='TEOS-10 5 day')
#plt.axvline(2013, c = 'red')
#plt.axvline(2018, c = 'red')
#plt.axvline(2004, c = 'red')
#plt.axvline(2005, c = 'red')
#plt.axvline(2006, c = 'red')
#plt.axvline(2007, c = 'red')

plt.xlabel('Year')
plt.ylabel('Mixed Layer Depth (m)')
plt.title('Max Yearly Mixed Layer Depth, EOS-80 & TEOS-10 5 day')
plt.gca().xaxis.set_major_locator(MaxNLocator(nbins=7))
plt.legend()

In [ ]:
plt.plot(years, teosm-eosm)

ok next do mean wintertime mld & map comparisons of 2005-2007, 2013, 2017, 2018, 2019
then send to paul
tell him you're working on arctic gateways